## Find the district name based on the latitude and longitude of the earthquake epicenter using the Nominatim API from OpenStreetMap.

#### NOTE BELOW ARE ARCHIEVED FILES. DO NOT USE THEM. THEY ARE KEPT FOR REFERENCE PURPOSES ONLY.<br/> AND IF U SEE UNDEFINED VARIABLES, THEY ARE DEFINED IN THE MAIN FILE. DO NOT WORRY ABOUT THEM.

In [ ]:
from enum import Enum
class Features(Enum):
	TIME = 'time'
	LATITUDE = 'latitude'
	LONGITUDE = 'longitude'
	DEPTH = 'depth'
	MAGNITUDE = 'mag'
	MAGNITUDE_TYPE = 'magType'
	NSTATIONS = 'nst'
	GAP = 'gap'
	DMIN = 'dmin'
	RMS = 'rms'
	HORIZONTAL_ERROR = 'horizontalError'
	DEPTH_ERROR = 'depthError'
	MAGNITUDE_ERROR = 'magError'
	MAGNITUDE_NSTATIONS = 'magNst'	

	DISTRICT = 'district'

df_filtered = df_original[[f.value for f in Features if f != Features.DISTRICT]]
df_filtered.head()

df_filtered.to_csv(FILTERED_DATA_PATH, index=False)

#### District Extraction 

In [ ]:

import requests
import time
import pandas as pd

class DistrictFinder:
	URL = "https://nominatim.openstreetmap.org/reverse"
	HEADERS = {
		"User-Agent": "my-nepal-district-app/1.0"
	}
	COUNTRY = "Nepal"
	SLEEP_TIME = 1  # Time to sleep between requests in seconds,to avoid hitting the rate limit of the API
	@staticmethod
	def get_district(lat, lon):
		url = DistrictFinder.URL
		headers = DistrictFinder.HEADERS
		params = {
			"lat": lat,
			"lon": lon,
			"format": "jsonv2",
			"addressdetails": 1,
			"accept-language": "en"
		}

		response = requests.get(
			url,
			params=params,
			headers=headers
		)

		data = response.json()

		address = data.get("address", {})
		country = address.get("country")
		district = address.get("county", "Unknown").replace(" District", "").strip()

		if country != DistrictFinder.COUNTRY:
			return "Unknown"
		else:
			return district
	@staticmethod

	def get_districts(latitudes:pd.Series , longitudes:pd.Series)-> pd.Series:
		districts = []
		for lat, lon in zip(latitudes, longitudes):
			district = DistrictFinder.get_district(lat, lon)
			districts.append(district)
			time.sleep(DistrictFinder.SLEEP_TIME) 
		return  pd.Series(districts, index=latitudes.index)

In [ ]:
import os
import pandas as pd

df_district = df_filtered.copy()

total_records = len(df_district)

print("Fetching districts for the earthquake data...\n"
	  "Grab a cup of coffee, this may take a while depending on the number of records.\n"
	  f"Estimated time: ~1 second per record. For {total_records} records," 
	  f" it will take {total_records // 60} minutes and {total_records % 60} seconds. at best\n"
	  "Most likely will take 2x of the extimated time due to network latency and API response time.\n")

TEMP_CONVERSION_COUNTING_FILE = 'data/temp_conversion_counting.txt'

# Make sure the target column exists, can  inject data into it
if Features.DISTRICT.value not in df_district.columns:
	df_district[Features.DISTRICT.value] = None

current_chunk_index = 0

# Check if we are resuming from a previous run
try:
	with open(TEMP_CONVERSION_COUNTING_FILE, 'r') as f:
		current_chunk_index = int(f.read().strip())
	
	# If resuming, load the existing backup CSV so we don't lose previous chunk data
	if current_chunk_index > 0 and os.path.exists(DISTRICT_DATA_PATH):
		df_district = pd.read_csv(DISTRICT_DATA_PATH)
		print(f"Resuming from saved backup: {DISTRICT_DATA_PATH}")
except FileNotFoundError:
	current_chunk_index = 0



total_records = len(df_district)
chunk_size = 25 
total_chunk_count = total_records // chunk_size + (1 if total_records % chunk_size != 0 else 0)

print(f"Starting from chunk index: {current_chunk_index}")
print(f"Total chunks: {total_chunk_count}")

# Loop through the remaining chunks
for i in range(current_chunk_index, total_chunk_count):
	print(f"Processing chunk {i + 1} of {total_chunk_count}...")
	
	# Calculate start and end indices for the current chunk
	start_idx = i * chunk_size
	end_idx = min((i + 1) * chunk_size, total_records)
	
	# Slice the latitudes and longitudes for this specific chunk
	chunk_lat = df_district[Features.LATITUDE.value].iloc[start_idx:end_idx]
	chunk_lon = df_district[Features.LONGITUDE.value].iloc[start_idx:end_idx]
	
	# Fetch districts
	chunk_districts = DistrictFinder.get_districts(chunk_lat, chunk_lon)
	
	# Assign the results back to the dataframe
	df_district.loc[df_district.index[start_idx:end_idx], Features.DISTRICT.value] = chunk_districts
	
	# Save the updated dataframe as a backup
	df_district.to_csv(DISTRICT_DATA_PATH, index=False)
	
	# Update the counting file to reflect the completed chunk
	with open(TEMP_CONVERSION_COUNTING_FILE, 'w') as f:
		f.write(str(i + 1))

print("All chunks processed successfully!")

# Clean up the temporary counting file since the process is completely finished
if os.path.exists(TEMP_CONVERSION_COUNTING_FILE):
	os.remove(TEMP_CONVERSION_COUNTING_FILE)

### Test dictionary

In [ ]:
earthquake_catalog = [
    {
        "name": "2015 Gorkha (Nepal)",
        "magnitude": 7.8,
        "depth_km": 15.0,
        "region": "active",
        "local_soil": "soft_basin",  # Gangetic Plain / Kathmandu basin alluvium
        "observed_felt_radius_km": "~800 - 1000"
    },
    {
        "name": "2011 Mineral (Virginia, USA)",
        "magnitude": 5.8,
        "depth_km": 6.0,
        "region": "stable_cena",
        "local_soil": "stiff_soil",  # Atlantic coastal plain sediments
        "observed_felt_radius_km": "~800 - 1000"
    },
    {
        "name": "2011 Tohoku (Japan)",
        "magnitude": 9.1,
        "depth_km": 29.0,
        "region": "subduction",
        "local_soil": "soft_basin",  # Kanto Plain / coastal basin sediments
        "observed_felt_radius_km": "> 1200"
    },
    {
        "name": "2023 Kahramanmaraş (Turkey)",
        "magnitude": 7.8,
        "depth_km": 10.0,
        "region": "strike_slip",
        "local_soil": "stiff_soil",  # Sedimentary basins across S. Turkey/Syria
        "observed_felt_radius_km": "~700 - 900"
    },
    {
        "name": "1989 Loma Prieta (California)",
        "magnitude": 6.9,
        "depth_km": 19.0,
        "region": "strike_slip",
        "local_soil": "soft_basin",  # SF Bay muds / Marina district alluvium
        "observed_felt_radius_km": "~350 - 450"
    },
    {
        "name": "2021 Fagradalsfjall (Iceland)",
        "magnitude": 5.7,
        "depth_km": 5.0,
        "region": "volcanic",
        "local_soil": "hard_rock",  # Dense basaltic lava bedrock
        "observed_felt_radius_km": "~100 - 150"
    },
    {
        "name": "2010 Christchurch (New Zealand)",
        "magnitude": 6.2,
        "depth_km": 5.0,
        "region": "active",
        "local_soil": "soft_basin",  # Canterbury alluvial gravels/silt
        "observed_felt_radius_km": "~250 - 350"
    }
]

TARGET_MMI = 2.5
results = []

#### V1

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import lambertw

def extract_seismic_features(
    df: pd.DataFrame, 
    felt_mmi: float = 2.5, 
    region: str = "active"
) -> pd.DataFrame:
    """
    Extracts physical felt-vibration features from earthquake catalog data.
    """
    data = df.copy()
    data['time'] = pd.to_datetime(data['time'])
    data = data.sort_values('time').reset_index(drop=True)
    
    # Regional parameters (USGS ShakeMap)
    if region == "active":
        c1, c2, c3, c4 = 2.08, 1.04, 2.09, 0.0019
    else:  # stable
        c1, c2, c3, c4 = 1.85, 1.15, 1.80, 0.0006
        
    M = data['mag'].values
    h = data['depth'].fillna(10.0).values  # Default to 10km depth if missing
    
    # 1. Near-field saturation radius R0(M)
    R0 = 10 ** (-0.281 + 0.251 * M)
    
    # 2. Transformed Lambert W parameters
    b = c3 / c4
    c = (c1 + c2 * M - felt_mmi) / c4
    
    # 3. Vectorized Lambert W evaluation for R_eff
    ln10 = np.log(10)
    z = (ln10 / b) * (10.0 ** (c / b))
    R_eff = (b / ln10) * lambertw(z).real
    
    # 4. Surface felt radius accounting for depth (h) and saturation (R0)
    # R_jb^2 = R_eff^2 - R0^2 - h^2
    rad_sq = R_eff**2 - R0**2 - h**2
    r_felt = np.where(rad_sq > 0, np.sqrt(rad_sq), 0.0)
    
    # 5. Extract core physical features
    data['r_felt_km'] = r_felt
    data['felt_area_sqkm'] = np.pi * (r_felt ** 2)
    
    # 6. Peak intensity directly above hypocenter (R_jb = 0)
    R_eff_epicenter = np.sqrt(R0**2 + h**2)
    data['max_nearfield_mmi'] = c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
    
    return data

# Example Usage with catalog data:
# df = pd.read_csv("earthquakes.csv")
# df_featured = extract_seismic_features(df)

####  Real Map visualizer.

In [ ]:
# import numpy as np
# import plotly.graph_objects as go

# def generate_circle_perimeter(lat: float, lon: float, radius_km: float, num_points: int = 40):
#     """Calculates geodesic perimeter lat/lon coordinates on a sphere."""
#     if radius_km <= 0:
#         return [], []
#     R = 6371.0  # Earth radius in km
#     angles = np.linspace(0, 2 * np.pi, num_points)
#     lat_rad, lon_rad = np.radians(lat), np.radians(lon)
#     d_R = radius_km / R

#     circle_lats = np.arcsin(
#         np.sin(lat_rad) * np.cos(d_R) + 
#         np.cos(lat_rad) * np.sin(d_R) * np.cos(angles)
#     )
#     circle_lons = lon_rad + np.arctan2(
#         np.sin(angles) * np.sin(d_R) * np.cos(lat_rad),
#         np.cos(d_R) - np.sin(lat_rad) * np.sin(circle_lats)
#     )
#     return np.degrees(circle_lats), np.degrees(circle_lons)


# fig = go.Figure()

# # 1. Epicenter points using go.Scattermap
# fig.add_trace(go.Scattermap(
#     lat=data_with_felt_features["latitude"],
#     lon=data_with_felt_features["longitude"],
#     mode="markers",
#     marker=dict(size=7, color="black"),
#     name="Epicenters",
#     text=data_with_felt_features["mag"],
#     hovertemplate="<b>Mag: %{text}</b><br>Lat: %{lat:.3f}<br>Lon: %{lon:.3f}<extra></extra>"
# ))

# # 2. True-scale physical circles
# for _, row in data_with_felt_features.iterrows():
#     r_km = row["felt_radius_km"]
#     if r_km > 0:
#         c_lats, c_lons = generate_circle_perimeter(row["latitude"], row["longitude"], r_km)
		
#         fig.add_trace(go.Scattermap(
#             lat=c_lats,
#             lon=c_lons,
#             mode="lines",
#             fill="toself",
#             fillcolor="rgba(239, 68, 68, 0.20)",
#             line=dict(color="darkred", width=1.5),
#             showlegend=False,
#             hoverinfo="skip"
#         ))

# # 3. Layout uses map= instead of mapbox=
# fig.update_layout(
#     map=dict(
#         style="open-street-map",  # Options: 'open-street-map', 'carto-positron', 'carto-darkmatter'
#         center=dict(lat=28.3949, lon=84.1240),
#         zoom=6.2,
#     ),
#     margin={"r": 0, "t": 40, "l": 0, "b": 0},
#     title="Nepal Earthquakes — Geodesic Felt Extent (km) Overlay"
# )

# fig.show()

### EarthQuake Data Finding From the Official web

In [ ]:
import requests
import time

EARTHQUAKE_IDS = [
	"us20002926",  # Nepal 2015
	"us20002b68",
	"us20002elh",
	"us20002a8d",
	"us6000pkpt",
	"usp00071m4",
	"us7000rkd1",
]

BASE_URL = (
	"https://earthquake.usgs.gov/fdsnws/event/1/query"
	"?eventid={event_id}&format=geojson"
)

for event_id in EARTHQUAKE_IDS:
	try:
		url = BASE_URL.format(event_id=event_id)

		response = requests.get(url, timeout=30)
		response.raise_for_status()

		data = response.json()

		properties = data.get("properties", {})
		products = properties.get("products", {})

		print(f"\n{'=' * 80}")
		print(f"Event ID: {event_id}")
		print(f"Magnitude: {properties.get('mag')}")
		print(f"Place: {properties.get('place')}")
		print(f"Available Products:")

		for product_name in products.keys():
			print(f"  - {product_name}")

		print(f"{'=' * 80}")

		time.sleep(0.5)

	except Exception as e:
		print(f"Failed: {event_id}")
		print(f"Error: {e}")

In [ ]:
import requests

event_id = "us20002926"

url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"
    f"?eventid={event_id}&format=geojson"
)

data = requests.get(url).json()

shakemap = data["properties"]["products"]["shakemap"][0]

mmi_url = shakemap["contents"]["download/cont_mmi.json"]["url"]

print(mmi_url)

mmi_data = requests.get(mmi_url).json()

print(mmi_data.keys())
print(mmi_data)

### V2

In [ ]:
import numpy as np
from scipy.special import lambertw

def compute_felt_radius_v2(
	magnitude: float,
	depth_km: float,
	target_mmi: float = 2.5,
	region: str = "active",
	soil_type: str = "solid_rock"
) -> float:
	"""
	Computes surface felt radius R_felt (km): the epicentral distance at which
	ground shaking is predicted to reach `target_mmi`, combining a regional
	tectonic attenuation model with a local soil/rock (Vs30) amplification term.
	"""
	# 1. Master Configuration Dictionaries
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}

	soil_configurations = {
		"hard_rock":   (1500.0, "Hard bedrock"),
		"solid_rock":  (760.0,  "Standard baseline engineering rock"),
		"soft_rock":   (560.0,  "Dense soil / weathered rock"),
		"stiff_soil":  (270.0,  "Stiff urban soil"),
		"soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
		"swamp_mud":   (100.0,  "Ultra-soft marsh muds")
	}

	# 2. Extract settings with safety defaults
	c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])
	vs30, _ = soil_configurations.get(soil_type, soil_configurations["solid_rock"])

	# 3. Site amplification relative to 760 m/s baseline rock
	v_ref = 760.0
	b_site = 1.8
	delta_mmi_site = b_site * np.log10(v_ref / vs30)

	# Back out the site term from the target: raises the required base-rock
	# intensity on hard rock (delta<0), lowers it on soft soil (delta>0)
	adjusted_target_mmi = target_mmi - delta_mmi_site

	# 4. Finite-fault saturation term
	R0 = 10 ** (-0.281 + 0.251 * magnitude)

	# 5. Lambert W setup for 3D effective distance
	a = c3 / c4
	b = (c1 + c2 * magnitude - adjusted_target_mmi) / c4

	ln10 = np.log(10)
	z = (ln10 / a) * (10.0 ** (b / a))
	R_eff = (a / ln10) * lambertw(z).real

	# 6. Pythagorean depth / finite-fault correction to surface distance
	rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

	if rad_sq <= 0:
		return 0.0

	return float(np.sqrt(rad_sq))

# --- Executing a Multivariable Grid Test ---
m, h, target = 7.8, 8.22, 3

print(f"--- MMI\u2265{target} Felt Radius Boundaries for a Mag {m} Event (Depth: {h}km) ---")
for soil in ["hard_rock", "solid_rock", "stiff_soil", "soft_basin"]:
	radius = compute_felt_radius_v2(m, h, target_mmi=target, region="active", soil_type=soil)
	print(f"-> Region: active | Rock Profile: {soil:<12} -> Radius: {radius:.1f} km")

In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.special import lambertw


@dataclass(frozen=True)
class FeltRadiusConfigV2:
	"""
	Configures the parameters for felt radius calculation.
	Allows customization of input/output column names, region type, and soil profile.
	Here, `i_` prefix denotes input columns, and `o_` prefix denotes output columns.
	Args:
		i_region: str - Tectonic region type (e.g., "active", "stable", "subduction", etc.)
		i_soil_type: str - Local soil/rock profile (e.g., "solid_rock", "soft_basin", etc.)
		i_time_column: str - Name of the time column in the DataFrame
		i_mag_column: str - Name of the magnitude column in the DataFrame
		i_depth_column: str - Name of the depth column in the DataFrame
		o_felt_radius_column: str - Name of the output felt radius column to be added to the DataFrame
		o_felt_area_column: str - Name of the output felt area column to be added to the DataFrame
		o_max_nearfield_mmi_column: str - Name of the output max near-field MMI column to be added to the DataFrame
	"""
	i_region: str = "active"
	i_soil_type: str = "solid_rock" 
	i_time_column: str = "time"
	i_mag_column: str = "mag"
	i_depth_column: str = "depth"
	o_felt_radius_column: str = "felt_radius_km"
	o_felt_area_column: str = "felt_area_sqkm"
	o_max_nearfield_mmi_column: str = "max_nearfield_mmi"


class FeltRadiusCalculatorV2:
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}
	supported_tectonic_regions = set(coefficient_configurations.keys())
	soil_configurations = {
		"hard_rock":   (1500.0, "Hard bedrock"),
		"solid_rock":  (760.0,  "Standard baseline engineering rock"),
		"soft_rock":   (560.0,  "Dense soil / weathered rock"),
		"stiff_soil":  (270.0,  "Stiff urban soil"),
		"soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
		"swamp_mud":   (100.0,  "Ultra-soft marsh muds")
	}
	supported_soil_profiles = set(soil_configurations.keys())

	@staticmethod
	def extract_seismic_features(
		df: pd.DataFrame, 
		felt_mmi: float = 2.5, 
		config: FeltRadiusConfigV2 = FeltRadiusConfigV2()
	) -> pd.DataFrame:
		"""
		Extracts physical felt-vibration features from earthquake catalog data
		incorporating tectonic attenuation and Vs30 local soil amplification.
		"""
		data = df.copy()
		
		# 1. Fetch and validate regional & soil configurations
		region = config.i_region
		soil_type = config.i_soil_type
		
		if region in FeltRadiusCalculatorV2.coefficient_configurations:
			c1, c2, c3, c4 = FeltRadiusCalculatorV2.coefficient_configurations[region]
		else:
			raise ValueError(f"Unknown region profile: {region}, supported: {FeltRadiusCalculatorV2.supported_tectonic_regions}")
			
		if soil_type in FeltRadiusCalculatorV2.soil_configurations:
			vs30, _ = FeltRadiusCalculatorV2.soil_configurations[soil_type]
		else:
			raise ValueError(f"Unknown soil profile: {soil_type}, supported: {FeltRadiusCalculatorV2.supported_soil_profiles}")
			
		M = data[config.i_mag_column].values
		h = data[config.i_depth_column].fillna(10.0).values  # Default to 10km depth if missing
		
		# 2. Site Amplification Correction (Vs30 vs 760m/s baseline)
		v_ref = 760.0
		b_site = 1.8
		delta_mmi_site = b_site * np.log10(v_ref / vs30)
		
		# Adjust target baseline intensity downward for soil amplification
		adjusted_target_mmi = felt_mmi - delta_mmi_site
		
		# 3. Near-field finite fault saturation term R0(M)
		R0 = 10 ** (-0.281 + 0.251 * M)
		
		# 4. Transformed Lambert W parameters
		a = c3 / c4
		b = (c1 + c2 * M - adjusted_target_mmi) / c4
		
		# 5. Vectorized Lambert W evaluation for R_eff across DataFrame arrays
		ln10 = np.log(10)
		z = (ln10 / a) * (10.0 ** (b / a))
		R_eff = (a / ln10) * lambertw(z).real
		
		# 6. Surface felt radius accounting for depth and saturation
		rad_sq = R_eff**2 - R0**2 - h**2
		r_felt = np.sqrt(np.maximum(rad_sq, 0.0))
		
		# 7. Append physical feature columns
		data[config.o_felt_radius_column] = r_felt
		data[config.o_felt_area_column] = np.pi * (r_felt ** 2)
		
		# 8. Peak intensity directly above hypocenter (R_jb = 0) with site correction
		R_eff_epicenter = np.sqrt(R0**2 + h**2)
		base_max_mmi = c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
		data[config.o_max_nearfield_mmi_column] = base_max_mmi + delta_mmi_site
		
		return data



config = FeltRadiusConfigV2(
	i_region="active",
	i_soil_type="soft_basin",
	i_time_column="time",
	i_mag_column="mag",
	i_depth_column="depth",
	o_felt_radius_column="felt_radius_km",
	o_felt_area_column="felt_area_sqkm",
	o_max_nearfield_mmi_column="max_nearfield_mmi"
)

mmi = 4.0
data_with_felt_features = FeltRadiusCalculatorV2.extract_seismic_features(
	df_filtered, 
	felt_mmi=mmi, 
	config=config
)

# data_with_felt_features.to_csv(INITIAL_EXTRACTION_DATA_PATH_TEMPLATE("2"), index=False)

### V3

In [ ]:
def calculate_site_amplification(
    vs30: float, 
    v_ref: float = 760.0, 
    b_site: float = 0.5, 
    upper_limit: float = 0.4, 
    lower_limit: float = -0.8, 
    method: str = "tanh"
) -> float:
    """
    Calculates MMI site amplification using continuous scaling functions.
    Supports both scalar values and NumPy arrays.
    """
    delta_raw = b_site * np.log10(v_ref / vs30)

    if method == "clip":
        # Legacy hard clipping (contains kink points / non-differentiable boundaries)
        delta_site = np.clip(delta_raw, lower_limit, upper_limit)

    elif method == "tanh":
        # Asymmetric smooth hyperbolic tangent saturation (C-infinity continuous)
        abs_lower = abs(lower_limit)
        delta_site = np.where(
            delta_raw >= 0,
            upper_limit * np.tanh(delta_raw / upper_limit),
            -abs_lower * np.tanh(np.abs(delta_raw) / abs_lower)
        )

    elif method == "sigmoid":
        # Generalized continuous logistic scaling
        span = upper_limit - lower_limit
        delta_site = lower_limit + span / (1.0 + np.exp(-4.0 * delta_raw / span))

    else:
        raise ValueError(f"Unknown site scaling method: {method}")

    return float(delta_site) if np.ndim(delta_site) == 0 else delta_site


def compute_felt_radius_v3(
    magnitude: float,
    depth_km: float,
    target_mmi: float = 2.5,
    region: str = "active",
    soil_type: str = "solid_rock",
    site_scaling_method: str = "tanh"
) -> float:
    """
    Computes surface felt radius R_felt (km): the epicentral distance at which
    ground shaking is predicted to reach `target_mmi`, combining a regional
    tectonic attenuation model with a local soil/rock (Vs30) continuous site amplification term.
    """
    # 1. Master Configuration Dictionaries
    coefficient_configurations = {
        "active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
        "stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
        "stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
        "subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
        "strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
        "volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
    }

    soil_configurations = {
        "hard_rock":   (1500.0, "Hard bedrock"),
        "solid_rock":  (760.0,  "Standard baseline engineering rock"),
        "soft_rock":   (560.0,  "Dense soil / weathered rock"),
        "stiff_soil":  (270.0,  "Stiff urban soil"),
        "soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
        "swamp_mud":   (100.0,  "Ultra-soft marsh muds")
    }

    # 2. Extract settings with safety defaults
    c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])
    vs30, _ = soil_configurations.get(soil_type, soil_configurations["solid_rock"])

    # 3. Continuous site amplification relative to 760 m/s baseline rock
    delta_mmi_site = calculate_site_amplification(
        vs30=vs30,
        v_ref=760.0,
        b_site=0.5,
        upper_limit=0.4,
        lower_limit=-0.8,
        method=site_scaling_method
    )

    adjusted_target_mmi = target_mmi - delta_mmi_site

    # 4. Finite-fault saturation term
    R0 = 10 ** (-0.281 + 0.251 * magnitude)

    # 5. Lambert W setup for 3D effective distance
    a = c3 / c4
    b = (c1 + c2 * magnitude - adjusted_target_mmi) / c4

    ln10 = np.log(10)
    z = (ln10 / a) * (10.0 ** (b / a))
    R_eff = (a / ln10) * lambertw(z).real

    # 6. Pythagorean depth / finite-fault correction to surface distance
    rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

    if rad_sq <= 0:
        return 0.0

    return float(np.sqrt(rad_sq))


# --- Executing Test Across All Rock/Soil Profiles ---
m, h, target = 7.8, 8.22, 3

print(f"--- MMI\u2265{target} Felt Radius Boundaries for a Mag {m} Event (Depth: {h}km) ---")
test_soils = ["hard_rock", "solid_rock", "soft_rock", "stiff_soil", "soft_basin", "swamp_mud"]

for soil in test_soils:
    radius = compute_felt_radius_v3(
        magnitude=m, 
        depth_km=h, 
        target_mmi=target, 
        region="active", 
        soil_type=soil, 
        site_scaling_method="tanh"
    )
    print(f"-> Region: active | Rock Profile: {soil:<12} -> Radius: {radius:.1f} km")

### V4

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import lambertw


def estimate_pga_rock_g(target_mmi: float) -> float:
    """
    Inverts the Worden et al. (2012) GMICE relationship to estimate reference rock 
    Peak Ground Acceleration (PGA_rock in g) from target intensity (MMI).
    """
    # Worden et al. (2012) MMI - PGA (cm/s^2) piecewise relationship
    if target_mmi <= 4.22:
        log10_pga_cms2 = (target_mmi - 1.78) / 3.66
    else:
        log10_pga_cms2 = (target_mmi - 0.59) / 5.51

    pga_cms2 = 10.0 ** log10_pga_cms2
    pga_g = pga_cms2 / 980.665  # Convert cm/s^2 to g
    return float(pga_g)


def compute_seyhan_stewart_b_site(
    vs30: float, 
    target_mmi: float = 2.5, 
    v_ref: float = 760.0
) -> float:
    """
    Computes b_site dynamically using published parameters from:
    1. Seyhan & Stewart (2014) [NGA-West2 non-linear site model for PGV / T ~ 1.0s]
    2. Worden et al. (2012) [GMICE conversion factor c_V = 3.47]
    """
    if np.isclose(vs30, v_ref) or vs30 >= v_ref:
        return 1.80  # Reference bedrock baseline

    # 1. Estimate rock PGA (g) required to produce target_mmi
    pga_rock_g = estimate_pga_rock_g(target_mmi)

    # 2. Seyhan & Stewart (2014) empirical constants for PGV / T ~ 1.0s
    vs30_star = min(vs30, v_ref)
    f1 = -0.60
    f3 = 0.10
    f4 = -0.21
    f5 = -0.007

    # Non-linear scaling parameter f2
    f2 = f4 * (np.exp(f5 * (vs30_star - 360.0)) - np.exp(f5 * (v_ref - 360.0)))

    # Natural log site amplification factor ln(F_site)
    ln_F_site = f1 * np.log(vs30_star / v_ref) + f2 * np.log((pga_rock_g + f3) / f3)

    # Convert natural log amplification to base-10
    log10_F_site = ln_F_site / np.log(10.0)

    # 3. Worden et al. (2012) GMICE slope for PGV (c_V = 3.47)
    delta_mmi_site = 3.47 * log10_F_site

    # 4. Extract effective b_site exponent: delta_mmi_site = b_site * log10(v_ref / vs30)
    log10_v_ratio = np.log10(v_ref / vs30)
    b_site = delta_mmi_site / log10_v_ratio

    return float(np.clip(b_site, 0.5, 2.5))


def compute_felt_radius_v4(
    magnitude: float,
    depth_km: float,
    target_mmi: float = 2.5,
    region: str = "active",
    soil_type: str = "solid_rock") -> float:
    """
    Computes surface felt radius R_felt (km): the epicentral distance at which
    ground shaking is predicted to reach `target_mmi`, combining regional
    tectonic attenuation with dynamic Seyhan & Stewart (2014) site amplification.
    """
    coefficient_configurations = {
        "active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
        "stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
        "stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
        "subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
        "strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
        "volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
    }

    soil_configurations = {
        "hard_rock":   (1500.0, "Hard bedrock"),
        "solid_rock":  (760.0,  "Standard baseline engineering rock"),
        "soft_rock":   (560.0,  "Dense soil / weathered rock"),
        "stiff_soil":  (270.0,  "Stiff urban soil"),
        "soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
        "swamp_mud":   (100.0,  "Ultra-soft marsh muds")
    }

    c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])
    vs30, _ = soil_configurations.get(soil_type, soil_configurations["solid_rock"])

    v_ref = 760.0

    # Dynamically compute b_site using published Seyhan & Stewart (2014) parameters
    b_site = compute_seyhan_stewart_b_site(
        vs30=vs30,
        target_mmi=target_mmi,
        v_ref=v_ref
    )

    delta_mmi_site = b_site * np.log10(v_ref / vs30)

    adjusted_target_mmi = target_mmi - delta_mmi_site

    R0 = 10 ** (-0.281 + 0.251 * magnitude)

    a = c3 / c4
    b = (c1 + c2 * magnitude - adjusted_target_mmi) / c4

    ln10 = np.log(10)
    z = (ln10 / a) * (10.0 ** (b / a))
    R_eff = (a / ln10) * lambertw(z).real

    rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

    if rad_sq <= 0:
        return 0.0

    return float(np.sqrt(rad_sq))